In [0]:
%pip install kaggle


In [0]:
username = "sonammagar"
api_key = "KGAT_48790b6f7eeb141e4cb2c1f6d67111ef"

In [0]:
import os
import json

# Replace with your actual Kaggle info
os.environ['KAGGLE_USERNAME'] = "sonammagar"
os.environ['KAGGLE_KEY'] = "KGAT_48790b6f7eeb141e4cb2c1f6d67111ef"

# Define a writable Workspace directory
user_email = "sonammagar143@gmail.com"
workspace_path = f"/Workspace/Users/{user_email}/kaggle_data"
os.makedirs(workspace_path, exist_ok=True)

# Set Kaggle to look in this Workspace folder for credentials
os.environ['KAGGLE_CONFIG_DIR'] = workspace_path

# Write the credentials file
with open(f"{workspace_path}/kaggle.json", "w") as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
os.chmod(f"{workspace_path}/kaggle.json", 0o600)


In [0]:
import zipfile

# Download the retail dataset to your Workspace
!kaggle datasets download -d arbaaztamboli/retail-markdown-optimization-discounts-and-sales --path {workspace_path}

# Unzip the file
zip_file = f"{workspace_path}/retail-markdown-optimization-discounts-and-sales.zip"
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(workspace_path)


In [0]:
import pandas as pd

# 1. Read with Pandas (Spark cannot read local Workspace files on shared clusters)
local_csv = f"{workspace_path}/SYNTHETIC Markdown Dataset.csv"
pdf = pd.read_csv(local_csv)

# 2. Convert to Spark and Clean Column Names (replace spaces with '_')
df = spark.createDataFrame(pdf)
clean_df = df.toDF(*(c.replace(' ', '_') for c in df.columns))

# 3. Drop existing table to avoid metadata mismatch, then save fresh
spark.sql("DROP TABLE IF EXISTS workspace.default.retail_markdown_final")
clean_df.write.mode("overwrite").saveAsTable("workspace.default.retail_markdown_final")

print("Ingestion Complete!")
display(spark.table("workspace.default.retail_markdown_final").limit(5))


In [0]:
%sql
SELECT * FROM workspace.default.retail_markdown_final LIMIT 10


In [0]:
%sql
SELECT * FROM workspace.default.retail_markdown_final